# 09i Is the personal map more than functional alignment? (THINGS-EEG1 development people)

**Kaggle:** CPU, Internet on. **Inputs:** the outputs of 08a and 08b (only the 22 development people are opened).

Development analysis, not a prespecified test. Each person borrows from the others. On the same k labelled images as the map: an orthogonal Procrustes map per
100 ms window and one for the whole epoch (hyperalignment-style, Haxby et al. 2011), and an SRM-style model (Chen et
al. 2015) with a shared 4- or 8-dimensional response learned from the others' predictions on 2,000 images.
k = 250 and 500 (draws 13 to 15) and 1,000 (draw 13).

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '09i-alignctl', 'cpu-dev'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"ks_three_draws": [250, 500], "ks_one_draw": [1000], "draws": [13, 14, 15], "window_s": [0.1, 0.2], "map_window_ms": 100, "srm_dims": [4, 8], "srm_fit_images": 2000, "srm_iters": 10, "srm_seed": 13, "methods": ["h4b", "procrustes_w100", "procrustes_epoch", "srm_d4", "srm_d8"], "dataset": "EEG1"}')
TAG, DATASET, DELTA = '09i', 'EEG1', 0.003547826037718968
PEOPLE = ['sub-04', 'sub-07', 'sub-11', 'sub-12', 'sub-13', 'sub-14', 'sub-15', 'sub-17', 'sub-20', 'sub-21', 'sub-24', 'sub-25', 'sub-26', 'sub-29', 'sub-30', 'sub-34', 'sub-35', 'sub-38', 'sub-39', 'sub-46', 'sub-48', 'sub-49']

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

from cross_subject_encoding.data import things_eeg2 as te
from cross_subject_encoding.eval import alignment_controls as al
from cross_subject_encoding.eval import eeg1_h4b as h4b
from cross_subject_encoding.eval import h4b_analysis as ha
from cross_subject_encoding.eval.paper_correlation import paper_correlation
from cross_subject_encoding.models import paper_linearizing as pl
from cross_subject_encoding.utils import config as C
from cross_subject_encoding.utils.kaggle_export import export_to_dataset

from cross_subject_encoding.data import things_eeg1 as e1
from cross_subject_encoding.utils.eeg1_pilot import sha256_file, source_file
assert sha256_file(C.find_artifact("eeg1_paper_kpca1000.npz")) == "61d36682db158fc622efe721c7759f70f52868a828aa9a66d395fdff7318ec1f"
with np.load(C.find_artifact("eeg1_paper_kpca1000.npz")) as f:
    z_train, z_test = f["train"].astype(np.float64), f["test"].astype(np.float64)
subsets = json.loads(C.find_artifact("eeg1_k_subsets.json").read_text())
people = sorted(PEOPLE)  # the 22 sources only; the 24 confirmation targets are never opened
y_train, bio_test = {}, {}
for s in people:
    for part in ("training", "test"):
        rel = te.load_part(source_file(Path("/kaggle/input"), s, part, people))
        if part == "training":
            y_train[s], times = rel["data"][:, 0], rel["times"]
        else:
            bio_test[s] = rel["data"]
order = lambda s, d, k: np.asarray(subsets["orders"][s][str(d)])[:k]
print(len(people), 'people loaded |', f'{(time.monotonic() - T0) / 60:.1f} min')

In [ ]:
ALPHAS = [10.0 ** e for e in range(-2, 9)]
N_TRAIN, N_TEST = len(z_train), len(z_test)
n_ch, n_t = 17, 100
query = np.concatenate([z_train, z_test])
src_train = np.empty((len(people), N_TRAIN, n_ch * n_t), np.float32)
src_test = np.empty((len(people), N_TEST, n_ch * n_t), np.float32)
for i, s in enumerate(people):
    pred = pl.ols_paper(z_train, y_train[s].reshape(N_TRAIN, -1), query)
    src_train[i], src_test[i] = pred[:N_TRAIN], pred[N_TRAIN:]
window = (times >= FIXED["window_s"][0]) & (times < FIXED["window_s"][1])
win = ha.window_index(times, FIXED["map_window_ms"])
win_epoch = ha.window_index(times, None)
fit_images = np.sort(np.random.default_rng(FIXED["srm_seed"]).permutation(N_TRAIN)[:FIXED["srm_fit_images"]])
plan = [(k, d) for k in FIXED["ks_three_draws"] for d in FIXED["draws"]] + [(k, FIXED["draws"][0]) for k in FIXED["ks_one_draw"]]
t3 = lambda a, n: np.asarray(a, np.float64).reshape(n, n_ch, n_t)
rows = []
for i, person in enumerate(people):
    t = time.monotonic()
    others = [j for j in range(len(people)) if j != i]
    avg_train, avg_test = src_train[others].mean(0), src_test[others].mean(0)
    avg_t3 = t3(avg_test, N_TEST)
    # the other people's SRM maps depend only on their own predictions, so they are fitted once per person
    srm_maps = {dd: al.srm_fit([t3(src_train[q][fit_images], len(fit_images)) for q in others], win, dd,
                               FIXED["srm_iters"], FIXED["srm_seed"]) for dd in FIXED["srm_dims"]}
    test_q = [t3(src_test[q], N_TEST) for q in others]
    preds = {"avg|all|all": avg_t3}
    for k, d in plan:
        idx = order(person, d, k)
        target = np.asarray(y_train[person][idx], np.float64)
        a_k = t3(avg_train[idx], k)
        coefs, _ = h4b.fit_map(a_k, target, win, seed=d)
        preds[f"h4b|{k}|{d}"] = h4b.apply_map(coefs, avg_t3, win)
        preds[f"procrustes_w100|{k}|{d}"] = al.apply_rotation(al.fit_procrustes_map(a_k, target, win), avg_t3, win)
        preds[f"procrustes_epoch|{k}|{d}"] = al.apply_rotation(al.fit_procrustes_map(a_k, target, win_epoch), avg_t3, win_epoch)
        src_k = [t3(src_train[q][idx], k) for q in others]
        for dd in FIXED["srm_dims"]:
            preds[f"srm_d{dd}|{k}|{d}"] = al.srm_apply(srm_maps[dd], src_k, test_q, target, win)
    preds = {n: np.asarray(p).reshape(N_TEST, n_ch, n_t) for n, p in preds.items()}
    if not all(np.isfinite(p).all() for p in preds.values()):
        raise ValueError(f"{person}: nonfinite prediction")
    scored = paper_correlation(preds, bio_test[person], n_iter=100)
    for name, curve in scored["correlation"].items():
        method, k, d = name.split("|")
        row = {"person": person, "method": method, "k": k, "draw": d, "r_100_200ms": float(curve[:, window].mean())}
        row.update({f"r_w{w}": float(curve[:, win == w].mean()) for w in np.unique(win)})  # per-window curves (rule 15)
        rows.append(row)
    pd.DataFrame(rows).to_csv(C.METRICS_DIR / f"{TAG}_scores.csv", index=False)
    print(person, f"{time.monotonic() - t:.0f} s |", f"{(time.monotonic() - T0) / 60:.1f} min")

In [ ]:
frame = pd.DataFrame(rows)
cols = ["r_100_200ms"] + sorted([c for c in frame.columns if c.startswith("r_w")], key=lambda c: int(c[3:]))
m = frame.groupby(["person", "method", "k"])[cols].mean()
ks = FIXED["ks_three_draws"] + FIXED["ks_one_draw"]
def per_person(method, k, col="r_100_200ms"):
    key = lambda p: (p, method, "all") if method == "avg" else (p, method, str(k))
    return np.array([m.loc[key(p), col] for p in people])
def compare(a, b):
    diff = a - b
    se = diff.std(ddof=1) / np.sqrt(len(diff))
    tcrit = stats.t.ppf(0.975, len(diff) - 1)
    return {"mean": float(diff.mean()), "lower95": float(diff.mean() - tcrit * se), "upper95": float(diff.mean() + tcrit * se),
            "positive": int((diff > 0).sum()), "n": int(len(diff))}
methods = ["avg"] + FIXED["methods"]
summary = {"people": len(people), "delta": DELTA, "means": {}, "h4b_minus": {}, "minus_avg": {}, "window_minus_avg_k500": {}}
for k in ks:
    summary["means"][k] = {meth: float(per_person(meth, k).mean()) for meth in methods}
    summary["h4b_minus"][k] = {meth: compare(per_person("h4b", k), per_person(meth, k)) for meth in methods if meth != "h4b"}
    summary["minus_avg"][k] = {meth: compare(per_person(meth, k), per_person("avg", k)) for meth in FIXED["methods"]}
for meth in FIXED["methods"]:
    summary["window_minus_avg_k500"][meth] = {f"{-200 + 100 * int(c[3:])} ms": float((per_person(meth, 500, c) - per_person("avg", 500, c)).mean()) for c in cols[1:]}
summary["rule"] = ("descriptive, development: H4b goes beyond hyperalignment if H4b minus Procrustes per window exceeds "
                   "delta with a 95% lower bound above 0 at k = 250 and 500; equivalent if the 95% interval lies within +-delta")
(C.METRICS_DIR / f"{TAG}_summary.json").write_text(json.dumps(summary, indent=2))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.9))
for meth in methods:
    ax[0].plot(ks, [summary["means"][k][meth] for k in ks], marker="o", label=meth)
ax[0].set(xscale="log", xlabel="labelled images k", ylabel="r, 100-200 ms", title=f"{DATASET}: H4b against alignment controls")
ax[0].legend(fontsize=7)
for meth in FIXED["methods"]:
    g = summary["window_minus_avg_k500"][meth]
    ax[1].plot([int(x.split()[0]) + 50 for x in g], list(g.values()), marker="o", label=meth)
ax[1].axhline(0, color="black", lw=0.5)
ax[1].set(xlabel="window centre (ms)", ylabel="gain over others' average", title="Per window, k = 500")
ax[1].legend(fontsize=7)
fig.tight_layout()
fig.savefig(C.METRICS_DIR / f"{TAG}_figure.png", dpi=150)
print(json.dumps({"means": summary["means"], "h4b_minus": summary["h4b_minus"]}, indent=1)[:4000])

In [ ]:
export_to_dataset(slug="cross-subject-brain-encoding-09i-outputs", title="XSUB 09i alignment controls outputs",
                  files=[C.METRICS_DIR / f"{TAG}_{x}" for x in ("summary.json", "scores.csv", "figure.png")],
                  include_dirs=[], message="functional-alignment controls against H4b")